# Assignment 2. Data Extraction and Statistical Hypothesis Testing

*Foundations of Data Science*  
*Dr. Khalaj (Fall 2025)*  

<!-- *For your questions refer to mehrshadtaji61@gmail.com or @m522t on Telegram* -->
In this exercise, you will collect real-world data from **TripAdvisor (tripadvisor.com)** to build a dataset of restaurants located in **New York City**.  
Your goal is to extract restaurant information from **six neighborhoods** and store the results in a structured **database** for later analysis.


### Information  
Complete the information box below.

Personal identifiers have been omitted from this public portfolio copy.


### Neighborhoods to Focus On

You will collect data from the following **six neighborhoods**:

1. **Downtown Manhattan (Downtown)**
2. **Midtown West**
3. **Midtown**
4. **Upper Manhattan (Uptown)**
5. **Upper East Side**
6. **Greenwich Village**

For each neighborhood, aim to extract data for approximately **100 restaurants**.

---

### Requirements and Setup

Your script **must** use `undetected_chromedriver` with **Selenium** to simulate human-like browsing and avoid being blocked by TripAdvisor’s protection system.

For each restaurant, extract the following fields:

| Field | Description |
|--------|--------------|
| **Name** | Restaurant name |
| **Neighborhood** | The area the restaurant belongs to |
| **Rating** | Average user rating |
| **Number of Reviews** | Total number of customer reviews |
| **Cuisine Type** | Main type(s) of food served (e.g., Italian, Japanese) |
| **Price Range** | Price level represented by `$`, `$$`, `$$$`, or `$$$$` |





### Anti-Blocking Guidelines

To minimize blocking or CAPTCHA issues, follow these guidelines:

- Always use **`undetected_chromedriver`** instead of the standard Chrome driver.  
- Add **random sleep intervals (2–5 seconds)** between actions.  
- **Scroll down** each page to ensure all restaurants are fully loaded.  
- If a **CAPTCHA** appears, **pause execution** and solve it manually.  
- Avoid sending requests or clicking too frequently.  
- Simulate realistic, human-like browsing behavior.

---

### Data Collecting


In [ ]:
#TODO
import time
import random
import csv
import re
import undetected_chromedriver as uc
from bs4 import BeautifulSoup


neighborhoods = [
    "Downtown Manhattan",
    "Midtown West",
    "Midtown",
    "Upper Manhattan",
    "Upper East Side",
    "Greenwich Village"
]

output_file = r"D:\Term5\Data Science\FDS_HW2\restaurants.csv"

def sleep(a=2, b=5):
    time.sleep(random.uniform(a, b))


def wait_for_captcha(driver):
    while True:
        html = driver.page_source.lower()

        blocked_keywords = [
            "access denied",
            "verify you are a human",
            "hcaptcha"
        ]

        if any(k in html for k in blocked_keywords):
            print("\n CAPTCHA detected!")
        else:
            break


def start_driver():
    options = uc.ChromeOptions()
    options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    return uc.Chrome(options=options)

def scroll(driver, steps=5):
    for _ in range(steps):
        driver.execute_script("window.scrollBy(0, 2000);")
        sleep(15, 20)


def scrape_all():
    driver = start_driver()
    restaurants = []
    urls = [
        "https://www.tripadvisor.com/FindRestaurants?geo=60763&offset={offset}&establishmentTypes=10591&neighborhoods=15565677&broadened=false",
        "https://www.tripadvisor.com/FindRestaurants?geo=60763&offset={offset}&establishmentTypes=10591&neighborhoods=15565657&broadened=false",
        "https://www.tripadvisor.com/FindRestaurants?geo=60763&offset={offset}&establishmentTypes=10591&neighborhoods=7102352&broadened=false",
        "https://www.tripadvisor.com/FindRestaurants?geo=60763&offset={offset}&establishmentTypes=10591&neighborhoods=15565675&broadened=false",
        "https://www.tripadvisor.com/FindRestaurants?geo=60763&offset={offset}&establishmentTypes=10591&neighborhoods=7102356&broadened=false",
        "https://www.tripadvisor.com/FindRestaurants?geo=60763&offset={offset}&establishmentTypes=10591&neighborhoods=7102348&broadened=false"
    ]

    for n_idx, url in enumerate(urls):
        for i in [0,30,60,90]:
            driver.get(url.format(offset=i))
            sleep(25, 55) 
            scroll(driver)
            html = driver.page_source
            soup = BeautifulSoup(html, "html.parser")
            cards = soup.find_all("div", attrs={"data-automation": "restaurantCard"})
            for card in cards:

                name_tag = card.find("div", class_=lambda x: x and "biGQs" in x)
                if name_tag:
                    name = name_tag.text.strip()
                    name = re.sub(r"^\d+\.\s*", "", name)
                print(name)

                rating_tag = card.find("svg", attrs={"data-automation":"bubbleRatingImage"})
                rating = ""
                if rating_tag:
                    title = rating_tag.find("title")
                    if title:
                        text = title.text.strip()     
                        rating = text.split(" ")[0]
                print(rating)

                reviews = ""
                rev_tag = card.find("div", {"data-automation": "bubbleReviewCount"})
                if rev_tag:
                    span = rev_tag.find("span")
                    if span:
                        text = span.text.strip()     
                        reviews = text.split()[0]             
                        reviews = reviews.replace(",", "")
                        reviews = reviews.replace("(", "")
                print(reviews)

                cuisine = ""
                price = ""
                block = card.find("div", class_=lambda c: c and "biqBm" in c)
                if not block:
                    return cuisine, price

                spans = block.find_all("span", class_=lambda c: c and "biGQs" in c)

                if len(spans) > 0:
                    cuisine = spans[0].get_text(strip=True)

                if len(spans) > 1:
                    price = spans[1].get_text(strip=True)
                print(cuisine)
                print(price)
                print(neighborhoods[n_idx])
                restaurants.append([name,neighborhoods[n_idx],rating,reviews,cuisine,price])
                sleep(2,4)

    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Name", "Neighborhood", "Rating", "Reviews", "Cuisine Type", "Price Range"])
        writer.writerows(restaurants)           





if __name__ == "__main__":
    scrape_all()

###  Data Storage

After collecting the data, store it in a **database** (not a flat file).  
You may use **SQLite** (recommended) or **MySQL/PostgreSQL**.



In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy.stats import ttest_ind
import numpy as np


In [ ]:
#TODO

csv_file = r"D:\Term5\Data Science\FDS_HW2\restaurants.csv"
db_file = r"D:\Term5\Data Science\FDS_HW2\restaurants.db"

df = pd.read_csv(csv_file)
conn_SQL = sqlite3.connect(db_file)
df.to_sql("restaurants", conn_SQL, if_exists="replace", index=False)

conn_SQL.close()


---

Using the data you collected, answer the following questions:

- How is the **distribution of restaurant prices** across different neighborhoods?  

The distribution of restaurant prices varies across neighborhoods. For example, in Midtown and the Upper East Side, price levels range approximately from 2.5 to 4 (after converting $ signs to numeric values). The other neighborhoods show almost no variation in price, with average price levels around 2.5.

- What are the **most common cuisine types** among restaurants?  

The top 15 most common cuisine types are shown below. The top 5 common ones are “American,” “Italian,” “French,” “Japanese,” and “Asian.”

- Do **Italian restaurants** show significant differences in their **average price levels** across neighborhoods in New York City?

There are no significant differences in average price levels of Italian restaurants across neighborhoods. However, as the plot shows, the average price is slightly higher in the Upper East Side and Midtown compared to other neighborhoods.

---


In [ ]:
#TODO
#How is the distribution of restaurant prices across different neighborhoods?

conn_SQL = sqlite3.connect("restaurants.db")
df = pd.read_sql_query('SELECT Neighborhood, "Price Range" FROM restaurants', conn_SQL)

price_mapping = {"$": 1, "$$": 2, "$$$": 3, "$$$$": 4}

def price_to_number(price_range):
    if pd.isna(price_range):
        return None
    parts = price_range.split('-')
    nums = [price_mapping[p.strip()] for p in parts if p.strip() in price_mapping]
    return sum(nums)/len(nums) if nums else None


df["price_value"] = df["Price Range"].apply(price_to_number)

#Box plot showing the price dirtribution
plt.figure(figsize=(10,6))
sns.boxplot(data=df, x="Neighborhood", y="price_value", order=df.groupby("Neighborhood")["price_value"].mean().sort_values(ascending=False).index)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Price Level")
plt.title("Distribution of Restaurant Prices Across Neighborhoods")
plt.show()

#Histogram
avg_price = df.groupby("Neighborhood")["price_value"].mean().sort_values(ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(x=avg_price.index, y=avg_price.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Average Price Level")
plt.title("Average Restaurant Price by Neighborhood")
plt.show()


#What are the most common cuisine types among restaurants?

df_cuisine = pd.read_sql_query('SELECT "Cuisine Type" FROM restaurants', conn_SQL)
df_exploded = df_cuisine['Cuisine Type'].str.split(',').explode().str.strip()
cuisine_counts = df_exploded.value_counts().head(15)

# Plot
plt.figure(figsize=(10,6))
cuisine_counts.plot(kind='barh', color='purple')
plt.xlabel("Number of Restaurants")
plt.ylabel("Cuisine Type")
plt.title("Top 15 Most Common Cuisine Types")
plt.gca().invert_yaxis()
plt.show()

#Do Italian restaurants show significant differences in their average price levels across neighborhoods in New York City?

df_italian = pd.read_sql_query('SELECT Neighborhood, "Price Range", "Cuisine Type" FROM restaurants', conn_SQL)
italian = df_italian[df_italian['Cuisine Type'].str.contains("Italian", case=False, na=False)]
italian['Price'] = italian['Price Range'].apply(price_to_number)
avg_price_italian = italian.groupby('Neighborhood')['Price'].mean().sort_values(ascending=False)

plt.figure(figsize=(10,6))
avg_price_italian.plot(kind='bar', color='limegreen')
plt.ylabel("Average Price Level")
plt.xlabel("Neighborhood")
plt.title("Average Price of Italian Restaurants by Neighborhood")
plt.xticks(rotation=45)
plt.show()


First, explore the **distribution of restaurant prices** across different neighborhoods.  
- **What patterns do you observe?**

The distribution of restaurant prices varies noticeably across neighborhoods. Midtown shows the highest average price levels, indicating that restaurants there tend to be more expensive than in other areas. In contrast, Upper Manhattan has lower average price levels, suggesting that restaurants in this neighborhood are generally more affordable compared to the rest of the city.

- **Are expensive restaurants concentrated in specific areas, or are they spread evenly across the city?**
  
Expensive restaurants are not evenly distributed across the city. They are concentrated in certain neighborhoods—particularly Midtown, which has the highest number of expensive restaurants. In contrast, Upper Manhattan has the fewest expensive restaurants, indicating that high-priced options are less common there.

Then, test the following **claim**:  

> *“Expensive restaurants ($$$–$$$$) have higher average ratings than cheaper ones ($–$$).”*  

A t-test comparing ratings of expensive and cheaper restaurants gave a t-statistic of 0.044 and a p-value of 0.965. Since the p-value is much higher than 0.05, there is no significant difference in average ratings between the two groups. Also a t near 0 means the group means are very close compared to the spread of the data.
This indicates that expensive restaurants do not necessarily have higher ratings than cheaper ones.

In [ ]:
#TODO
#distribution of restaurant prices across different neighborhoods
conn_SQL = sqlite3.connect("restaurants.db")
df = pd.read_sql_query('SELECT Neighborhood, "Price Range" FROM restaurants', conn_SQL)

price_mapping = {"$": 1, "$$": 2, "$$$": 3, "$$$$": 4}

def price_to_number(price_range):
    if pd.isna(price_range):
        return None
    parts = price_range.split('-')
    nums = [price_mapping[p.strip()] for p in parts if p.strip() in price_mapping]
    return sum(nums)/len(nums) if nums else None

df["price_value"] = df["Price Range"].apply(price_to_number)

#Histogram
avg_price = df.groupby("Neighborhood")["price_value"].mean().sort_values(ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(x=avg_price.index, y=avg_price.values,color="pink")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Average Price Level")
plt.title("Average Restaurant Price by Neighborhood")
plt.show()


#Are expensive restaurants concentrated in specific areas, or are they spread evenly across the city?  
expensive_restaurants = df[df['price_value'] >= 3]
expensive_count = expensive_restaurants.groupby('Neighborhood').size().sort_values(ascending=False)

plt.figure(figsize=(10,6))
expensive_count.plot(kind='bar', color='goldenrod')
plt.ylabel("Number of Expensive Restaurants")
plt.xlabel("Neighborhood")
plt.title("Distribution of Expensive Restaurants Across Neighborhoods")
plt.xticks(rotation=45)
plt.show()


#Expensive restaurants ($$$–$$$$) have higher average ratings than cheaper ones ($–$$)  
df_claim = pd.read_sql_query('SELECT "Price Range", "Rating" FROM restaurants', conn_SQL)
df_claim['Price'] = df_claim['Price Range'].apply(price_to_number)
def rating(r):
    try:
        return float(r)
    except:
        return None 

df_claim['Rating_value'] = df_claim['Rating'].apply(rating)

# Expensive
expensive = df_claim[df_claim['Price'] >= 3]['Rating_value'].dropna()
# Cheaper
cheaper = df_claim[df_claim['Price'] < 3]['Rating_value'].dropna()

avg_expensive = expensive.mean()
avg_cheaper = cheaper.mean()

print("Average rating of expensive restaurants:", avg_expensive)
print("Average rating of cheaper restaurants:", avg_cheaper)

#plot
plt.figure(figsize=(8,5))
plt.boxplot([cheaper, expensive], labels=["cheaper", "expensive"], patch_artist=True,boxprops=dict(facecolor='skyblue'), medianprops=dict(color='red'))
plt.ylabel("Rating")
plt.title("Comparison of Ratings: Expensive vs Cheaper Restaurants")
plt.show()

# t_test and p_value
t_stat, p_val = ttest_ind(expensive, cheaper, equal_var=False)  # Welch's t-test
print("t-statistic:", t_stat)
print("p-value:", p_val)



Now focus on restaurant ratings in two different neighborhoods (for example, Downtown Manhattan vs. Upper Manhattan).

Test the following hypothesis:

- H₀: The average rating of restaurants in both neighborhoods is equal.
- H₁: Restaurants in Downtown Manhattan have higher average ratings than those in Upper Manhattan.

**Since the obtained p-value is 0.0824, which is larger than the 0.05 significance level, we fail to reject the null hypothesis. Therefore, there is not enough statistical evidence at the 5% level to conclude that Downtown Manhattan has higher restaurant ratings than Upper Manhattan.**

**The t-statistic is 1.39, which indicates that the average rating in Downtown Manhattan is slightly higher than in Upper Manhattan, but the difference is too small relative to the variation in the data to be considered statistically significant.**

In [ ]:
#TODO

conn_SQL = sqlite3.connect("restaurants.db")
df = pd.read_sql_query('SELECT Neighborhood, Rating FROM restaurants', conn_SQL)

def rating(r):
    try:
        return float(r)
    except:
        return None

df['Rating_value'] = df['Rating'].apply(rating)
df = df.dropna(subset=['Rating_value'])

downtown = df[df['Neighborhood'] == "Downtown Manhattan"]['Rating_value']
upper = df[df['Neighborhood'] == "Upper Manhattan"]['Rating_value']

# Two-sample t test 
t_stat, p_val_two_sided = ttest_ind(downtown, upper, equal_var=False)

p_val_one_sided = p_val_two_sided / 2 if t_stat > 0 else 1 - (p_val_two_sided / 2)

print("Downtown Manhattan mean:", downtown.mean())
print("Upper Manhattan mean:", upper.mean())
print("t-statistic:", t_stat)
print("one-sided p-value:", p_val_one_sided)


In this section, we want to explore whether restaurants with higher ratings also have more visitors.
Our dataset does not directly contain the number of visitors, but according to studies on restaurant review platforms, only about 7% of customers actually leave a review.


Create a scatter plot between rating and log(reviews + 1).

In [ ]:
#TODO

conn_SQL = sqlite3.connect("restaurants.db")
df = pd.read_sql_query('SELECT Rating, Reviews FROM restaurants', conn_SQL)

df = df.dropna(subset=['Rating', 'Reviews'])

df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')
df = df.dropna(subset=['Reviews'])

df['log_reviews'] = np.log(df['Reviews'] + 1)
df = df.sort_values(by="Rating", ascending=True)
# Scatter plot
plt.figure(figsize=(8,6))
plt.scatter(df['Rating'], df['log_reviews'], alpha=0.6)

plt.xlabel("Rating")
plt.ylabel("log(Reviews + 1)")
plt.title("Relationship between Rating and Number of visitors ")

plt.grid(True)
plt.show()


Calculate the correlation coefficient between these two variables.

Do restaurants with higher ratings usually have more reviews?
Does this relationship appear linear?

>write your anwser here

The correlation coefficient is calculated as -0.046, which is very close to zero, indicating that **there is no significant linear relationship** between ratings and reviews. As observed in the figure above, restaurants with ratings between 4 and 4.5 tend to have the most reviews, while restaurants with higher or lower ratings generally have fewer reviews. The negative sign suggests that, very slightly, higher ratings are associated with fewer reviews, but this effect is negligible. Overall, the data do not support the conclusion that restaurants with higher ratings necessarily have more reviews or visitors.

In [ ]:
#TODO
conn_SQL = sqlite3.connect("restaurants.db")
df_corr = pd.read_sql_query('SELECT Rating, Reviews FROM restaurants', conn_SQL)

df_corr['Reviews'] = pd.to_numeric(df_corr['Reviews'], errors='coerce')
df_corr['Rating'] = pd.to_numeric(df_corr['Rating'], errors='coerce')
df_corr = df_corr.dropna(subset=['Rating', 'Reviews'])
df_corr['log_reviews'] = np.log(df_corr['Reviews']+1)

#correlation
corr = df_corr['Rating'].corr(df_corr['log_reviews'])
print("Correlation coefficient:", corr)

Could neighborhood be a factor influencing  the number of reviews?

In [ ]:
#TODO
conn_SQL = sqlite3.connect("restaurants.db")
df = pd.read_sql_query('SELECT Neighborhood, Reviews FROM restaurants', conn_SQL)
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')
df = df.dropna(subset=['Reviews'])
df['log_reviews'] = np.log(df['Reviews']+1)
avg_reviews = df.groupby('Neighborhood')['log_reviews'].mean()

# Boxplot
plt.figure(figsize=(10,6))
df.boxplot(column='log_reviews', by='Neighborhood', grid=False)
plt.ylabel('Number of Reviews')
plt.title('Distribution of Reviews by Neighborhood')
plt.suptitle('') 
plt.xticks(rotation=45)
plt.show()


Now analyze the relationship between rating, number of visits (approximated by reviews), and neighborhood.

**As the graph shows, Midtown has the highest number of reviews compared to other neighborhoods, and its restaurants are generally rated between 4 and 4.75. On the other hand, Upper Manhattan has the fewest reviews and, presumably, the least number of visitors, with ratings ranging from 3.75 to 4.5. The other four neighborhoods fall somewhere in between. For example, Upper East Side and Greenwich Village show similar patterns, with average reviews around 5 to 7 (log scale) and ratings mostly between 4 and 4.5.**

In [ ]:
conn_SQL = sqlite3.connect("restaurants.db")
df = pd.read_sql_query('SELECT Neighborhood, Reviews, Rating FROM restaurants', conn_SQL)
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df = df.dropna(subset=['Rating', 'Reviews'])
df['log_reviews'] = np.log(df['Reviews']+1)

plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='log_reviews', y='Rating', hue='Neighborhood')
plt.xlabel('Number of Reviews (log scale)')
plt.ylabel('Rating')
plt.title('Ratings vs. Number of Reviews by Neighborhood')
plt.show()
